# btp-student-pilot
Inspect the compact architecture and run a two-update CPU plumbing test. Real training requires an audited corpus and an approved cumulative budget.

Implementation workflow; no completed GPU experiment is implied. Historical notebooks are preserved.

## Source and runtime
Attach the private research source bundle. Internet is needed to provision the isolated runtime; running this cell downloads dependencies. CPU smoke precedes GPU use. No long experiment runs automatically.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

ROOT = Path("/kaggle/working/BTP")
OUTPUT = Path("/kaggle/working") / ("btp-research-" + str(time.time_ns()))
OUTPUT.mkdir()
# Upload artifacts/btp-kaggle.zip as btp-research-source.bundle in a private dataset.
bundles = list(Path("/kaggle/input").rglob("btp-research-source.bundle"))
if not ROOT.exists():
    assert len(bundles) == 1, (
        "Attach one research source bundle; see docs/RESEARCH_WORKFLOWS.md"
    )
    with zipfile.ZipFile(bundles[0]) as archive:
        for info in archive.infolist():
            destination = (ROOT.parent / info.filename).resolve()
            assert destination.is_relative_to(ROOT), "Unsafe source member"
        archive.extractall(ROOT.parent)
snapshot = json.loads((ROOT / "source-manifest.json").read_text())
for name, expected in snapshot["files"].items():
    path = (ROOT / name).resolve()
    assert path.is_relative_to(ROOT) and path.is_file(), name
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, name
print("Verified source files:", len(snapshot["files"]))

In [ ]:
uv = shutil.which("uv")
if uv is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
UV = [uv] if uv else [sys.executable, "-m", "uv"]
PYTHON = Path("/tmp/btp-research-env/bin/python")
if not PYTHON.exists():
    subprocess.run(
        UV + ["venv", "--python", "3.10", str(PYTHON.parent.parent)], check=True
    )
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "torch==2.6.0",
        "torchvision==0.21.0",
        "--index-url",
        "https://download.pytorch.org/whl/cu118",
    ],
    check=True,
)
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "-r",
        str(ROOT / "configs/kaggle-requirements.txt"),
    ],
    check=True,
)
RUN = UV + ["run", "--no-project", "--python", str(PYTHON), "python"]
ENVIRONMENT = dict(os.environ, PYTHONPATH=str(ROOT / "src"), CUDA_VISIBLE_DEVICES="0")


def run(arguments):
    subprocess.run(RUN + arguments, cwd=ROOT, env=ENVIRONMENT, check=True)


with (OUTPUT / "runtime-packages.txt").open("w") as handle:
    subprocess.run(
        UV + ["pip", "freeze", "--python", str(PYTHON)], stdout=handle, check=True
    )
(OUTPUT / "source-manifest.json").write_text(json.dumps(snapshot, indent=2))

In [ ]:
run(["scripts/smoke_student.py"])

In [ ]:
TRAIN_MANIFEST = None
SOURCE_COMMIT = None  # Exact Image-Dehazing commit used for this source bundle
RUN_TRAINING = False
RESUME_CHECKPOINT = None
UNTIL_UPDATE = 100  # Absolute endpoint; first profile counts inside total pilot cap
if TRAIN_MANIFEST is not None:
    assert SOURCE_COMMIT, "Record the exact project source commit"
    arguments = [
        "-m",
        "btp.student.train_cli",
        "--manifest",
        str(TRAIN_MANIFEST),
        "--data-root",
        str(Path(TRAIN_MANIFEST).parent),
        "--evaluation-manifest",
        str(ROOT / "configs/evaluation/nh_haze_manifest.json"),
        "--protocol",
        str(ROOT / "configs/research/protocol-v1.json"),
        "--architecture",
        str(ROOT / "configs/research/student/architecture.json"),
        "--source-commit",
        SOURCE_COMMIT,
        "--output",
        str(OUTPUT / "pilot"),
        "--until-update",
        str(UNTIL_UPDATE),
        "--device",
        "cuda:0",
    ]
    if RUN_TRAINING:
        arguments.append("--run")  # Enable only after explicit budget approval
    if RESUME_CHECKPOINT is not None:
        arguments.extend(["--resume", str(RESUME_CHECKPOINT)])
    run(arguments)
else:
    print("No training corpus supplied; fixture only.")

## Pilot gate
The smoke uses random width-two fixture weights, not a dehazing checkpoint. The default width-16 architecture has 1,724,275 parameters. Use `BoundedTrainer` only after corpus audit; count the first 100 profiling updates inside the proposed 2,000-update/two-T4-hour cap. See the training recipe in docs/RESEARCH_WORKFLOWS.md. No actual training corpus or budget approval is bundled here.

In [ ]:
archive = shutil.make_archive(str(OUTPUT), "zip", OUTPUT)
print("Download evidence:", archive)